# Module 13 – Probability
## Objective
Use Python simulations to explore probability, Bayes' theorem, random variables, continuous distributions, and the Central Limit Theorem.

## 1. Basics of Probability
We simulate 10,000 coin tosses and two-dice rolls. Experimental probability is the number of successful outcomes divided by the total number of trials.

In [ ]:
import random

random.seed(42)  # Reproducible results

# A. Coin toss simulation
tosses = []
for _ in range(10_000):
    tosses.append(random.choice(["Heads", "Tails"]))

heads_count = tosses.count("Heads")
tails_count = tosses.count("Tails")
print("Coin tosses: 10,000")
print(f"Heads: {heads_count}; experimental probability = {heads_count / len(tosses):.4f}")
print(f"Tails: {tails_count}; experimental probability = {tails_count / len(tosses):.4f}")

# B. Roll two fair dice and estimate P(sum = 7)
dice_trials = 10_000
sum_seven_count = 0
for _ in range(dice_trials):
    die1 = random.randint(1, 6)
    die2 = random.randint(1, 6)
    if die1 + die2 == 7:
        sum_seven_count += 1

print(f"\nTwo-dice trials: {dice_trials}")
print(f"Sum of 7 count: {sum_seven_count}")
print(f"Experimental P(sum = 7): {sum_seven_count / dice_trials:.4f}")
print(f"Theoretical P(sum = 7): {6 / 36:.4f}")

## 2. Probability of at Least One Six in Ten Rolls
The function below simulates ten rolls per trial. A trial succeeds if at least one roll is a six. We compare the estimate with `1 - (5/6)**10`.

In [ ]:
def estimate_at_least_one_six(number_of_trials=10_000):
    successful_trials = 0
    for _ in range(number_of_trials):
        got_six = False
        for _ in range(10):
            if random.randint(1, 6) == 6:
                got_six = True
                break
        if got_six:
            successful_trials += 1
    return successful_trials / number_of_trials

estimated_probability = estimate_at_least_one_six()
theoretical_probability = 1 - (5 / 6) ** 10
print(f"Estimated probability: {estimated_probability:.4f}")
print(f"Theoretical probability: {theoretical_probability:.4f}")

## 3. Conditional Probability and Bayes' Theorem
The bag contains 5 red, 7 green, and 8 blue balls. Draws are with replacement, so consecutive draws are independent. Let A be 'current draw is red' and B be 'previous draw is blue'. We estimate P(A|B), then verify Bayes' theorem from simulated event counts.

In [ ]:
from collections import Counter

random.seed(43)
bag = ["Red"] * 5 + ["Green"] * 7 + ["Blue"] * 8
number_of_pairs = 100_000
pairs = []

for _ in range(number_of_pairs):
    previous_color = random.choice(bag)
    current_color = random.choice(bag)  # replacement means each draw has the same probabilities
    pairs.append((previous_color, current_color))

previous_blue = sum(1 for prev, curr in pairs if prev == "Blue")
current_red = sum(1 for prev, curr in pairs if curr == "Red")
blue_then_red = sum(1 for prev, curr in pairs if prev == "Blue" and curr == "Red")

p_a = current_red / number_of_pairs
p_b = previous_blue / number_of_pairs
p_a_and_b = blue_then_red / number_of_pairs
p_a_given_b = blue_then_red / previous_blue if previous_blue else float("nan")

# P(B|A): among pairs whose current draw is red, fraction with previous draw blue
p_b_given_a = blue_then_red / current_red if current_red else float("nan")
bayes_estimate = (p_b_given_a * p_a / p_b) if p_b else float("nan")

print(f"Estimated P(current red | previous blue): {p_a_given_b:.4f}")
print(f"Estimated P(current red): {p_a:.4f}")
print(f"Estimated P(previous blue): {p_b:.4f}")
print(f"Estimated P(previous blue and current red): {p_a_and_b:.4f}")
print(f"Bayes theorem estimate P(A|B): {bayes_estimate:.4f}")
print("Theoretical P(current red | previous blue):", 5 / 20)
print("\nThe two estimates should be close, with small differences due to random sampling.")

## 4. Discrete Random Variable
Generate 1,000 observations where X takes values 1, 2, or 3 with the probabilities specified in the assignment. NumPy's `choice` samples according to those probabilities.

In [ ]:
import numpy as np

np.random.seed(42)
values = [1, 2, 3]
probabilities = [0.25, 0.35, 0.40]
sample = np.random.choice(values, size=1000, p=probabilities)

print("First 20 observations:", sample[:20])
print(f"Empirical mean: {np.mean(sample):.4f}")
print(f"Empirical variance: {np.var(sample):.4f}")
print(f"Empirical standard deviation: {np.std(sample):.4f}")

theoretical_mean = sum(x * p for x, p in zip(values, probabilities))
theoretical_variance = sum(p * (x - theoretical_mean) ** 2 for x, p in zip(values, probabilities))
print(f"Theoretical mean: {theoretical_mean:.4f}")
print(f"Theoretical variance: {theoretical_variance:.4f}")

## 5. Continuous Random Variable: Exponential Distribution
An exponential distribution with mean 5 has scale parameter 5. The histogram is normalized as a density so it can be compared with the theoretical PDF.

In [ ]:
import matplotlib.pyplot as plt

np.random.seed(42)
mean_value = 5
exponential_sample = np.random.exponential(scale=mean_value, size=2000)

x = np.linspace(0, exponential_sample.max(), 400)
pdf = (1 / mean_value) * np.exp(-x / mean_value)

plt.figure(figsize=(9, 5))
plt.hist(exponential_sample, bins=30, density=True, alpha=0.65, edgecolor="black", label="Simulated sample")
plt.plot(x, pdf, linewidth=2, label="Theoretical exponential PDF")
plt.title("Exponential Distribution (Mean = 5)")
plt.xlabel("Value")
plt.ylabel("Density")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

print(f"Sample mean (expected to be near 5): {np.mean(exponential_sample):.3f}")

## 6. Central Limit Theorem
First generate 10,000 values from a uniform distribution. Then repeatedly draw samples of size 30 from this population and calculate each sample mean. The sample means should form a more bell-shaped distribution than the original uniform population.

In [ ]:
np.random.seed(42)

population = np.random.uniform(low=0, high=1, size=10_000)
number_of_samples = 1000
sample_size = 30
sample_matrix = np.random.choice(population, size=(number_of_samples, sample_size), replace=True)
sample_means = sample_matrix.mean(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].hist(population, bins=30, density=True, edgecolor="black", alpha=0.7)
axes[0].set_title("Uniform Population")
axes[0].set_xlabel("Value")
axes[0].set_ylabel("Density")

axes[1].hist(sample_means, bins=30, density=True, edgecolor="black", alpha=0.7)
axes[1].set_title("Distribution of 1,000 Sample Means (n = 30)")
axes[1].set_xlabel("Sample mean")
axes[1].set_ylabel("Density")

plt.tight_layout()
plt.show()

print(f"Population mean: {np.mean(population):.4f}")
print(f"Mean of sample means: {np.mean(sample_means):.4f}")
print(f"Standard deviation of sample means: {np.std(sample_means):.4f}")

## Conclusion
Simulation results will vary slightly between runs unless random seeds are set. Experimental probabilities approach theoretical probabilities as the number of trials increases. The distribution of sample means illustrates the Central Limit Theorem.